In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

In [4]:
import os
import json
import pandas as pd
from PIL import Image
import unicodedata  # 0번 섹션에 추가 필요
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision.transforms import v2
import yaml


In [5]:
DEVICE =torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/") # unzipped folder

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR = os.path.join(extract_path, "train_images")
TEST_IMG_DIR  = os.path.join(extract_path, "test_images")

In [7]:
with open(TRAIN_JSON_PATH, 'r') as f:
    train_data = json.load(f)

In [8]:
with open(TEST_JSON_PATH, 'r') as f:
    test_data = json.load(f)

In [9]:
def build_df_from_merged_json(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    id_to_fname = {img["id"]: img["file_name"] for img in data["images"]}

    records = []

    for ann in data["annotations"]:
        img_id_coco = ann["image_id"]
        file_name = id_to_fname.get(img_id_coco)
        
        if file_name is None: continue
        
        img_path = os.path.join(img_dir, file_name)
        
        # 실제 이미지 파일이 있는지 확인 (선택 사항이지만 안전함)
        if not os.path.exists(img_path):
            continue

        x, y, w, h = ann["bbox"]
        
        records.append({
            "image_path": img_path,
            "image_id": os.path.splitext(file_name)[0], # 파일명을 ID로 사용
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

    return pd.DataFrame(records)

df= build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
print(f" train 데이터: {len(df)}개 객체")
print(f" train 이미지: {df['image_id'].nunique()}장")



 train 데이터: 4526개 객체
 train 이미지: 1489장


In [10]:
#  이상 bbox 사전 제거
IMG_W, IMG_H = 1280, 1280

before = len(df)
df = df[
    (df["bbox_w"] > 1) &
    (df["bbox_h"] > 1) &
    (df["bbox_x"] >= 0) &
    (df["bbox_y"] >= 0) &
    (df["bbox_x"] + df["bbox_w"] <= IMG_W) &
    (df["bbox_y"] + df["bbox_h"] <= IMG_H)
].reset_index(drop=True)

print(f"제거된 bbox: {before - len(df)}개 | 남은 bbox: {len(df)}개")

제거된 bbox: 2개 | 남은 bbox: 4524개


In [11]:
import random
import pandas as pd

# 랜덤 시드 고정
random.seed(42)

# 1) 이미지 ID 리스트 가져오기
unique_image_ids = df["image_id"].unique().tolist()
random.shuffle(unique_image_ids)

# 2) 9:1 비율로 split
split_idx = int(0.9 * len(unique_image_ids))
train_ids = unique_image_ids[:split_idx]
val_ids   = unique_image_ids[split_idx:]

# 3) 이미지 ID 기준으로 데이터프레임 분할
train_df = df[df["image_id"].isin(train_ids)].reset_index(drop=True)
val_df   = df[df["image_id"].isin(val_ids)].reset_index(drop=True)

print(f"✅ Train 데이터: {len(train_df)}개 객체, {train_df['image_id'].nunique()}장 이미지")
print(f"✅ Val 데이터:   {len(val_df)}개 객체, {val_df['image_id'].nunique()}장 이미지")

✅ Train 데이터: 4061개 객체, 1340장 이미지
✅ Val 데이터:   463개 객체, 149장 이미지


In [12]:
# -----------------------------
# 3) 클래스 매핑
# -----------------------------
def build_class_mappings(df):
    unique_cats = sorted(df['category_id'].unique())
    orig2model_yolo = {cid: i for i, cid in enumerate(unique_cats)}
    model2orig_yolo = {v: k for k, v in orig2model_yolo.items()}
    num_classes = len(orig2model_yolo)
    print(f"✅ 클래스 수: {num_classes}")
    return orig2model_yolo, model2orig_yolo, num_classes, unique_cats

In [13]:
unique_cats = sorted(df["category_id"].unique())
print(len(unique_cats))

orig2model_yolo = {cid: i for i, cid in enumerate(unique_cats)}  # without background
model2orig_yolo = {v: k for k, v in orig2model_yolo.items()}
num_classes = len(orig2model_yolo)  # it should be 73
print(num_classes)

73
73


In [14]:
# -----------------------------
# 4) YOLO 라벨 생성 (벡터화)
# -----------------------------
def save_yolo_labels_fast(df, output_dir, orig2model_yolo, img_size=1280):
    os.makedirs(output_dir, exist_ok=True)
    df = df.copy()
    df['cls'] = df['category_id'].map(orig2model_yolo)
    df['x_center'] = (df['bbox_x'] + df['bbox_w']/2) / img_size
    df['y_center'] = (df['bbox_y'] + df['bbox_h']/2) / img_size
    df['w_n'] = df['bbox_w'] / img_size
    df['h_n'] = df['bbox_h'] / img_size

    for img_id, group in tqdm(df.groupby('image_id'), desc=f"Saving labels to {output_dir}"):
        lines = "\n".join(
            f"{int(r.cls)} {r.x_center:.6f} {r.y_center:.6f} {r.w_n:.6f} {r.h_n:.6f}"
            for r in group.itertuples()
        )
        with open(os.path.join(output_dir, f"{img_id}.txt"), "w") as f:
            f.write(lines + "\n")

In [15]:
import os

os.makedirs("/content/yolo_dataset/images/train", exist_ok=True)
os.makedirs("/content/yolo_dataset/images/val", exist_ok=True)
os.makedirs("/content/yolo_dataset/labels/train", exist_ok=True)
os.makedirs("/content/yolo_dataset/labels/val", exist_ok=True)

In [16]:
import shutil
from tqdm import tqdm

# train 이미지 복사
for img_file in tqdm(train_df['image_path'].unique(), desc="Copying train images"):
    if os.path.exists(img_file):
        shutil.copy(img_file, "/content/yolo_dataset/images/train")
    else:
        print("Missing train image:", img_file)

# val 이미지 복사
for img_file in tqdm(val_df['image_path'].unique(), desc="Copying val images"):
    if os.path.exists(img_file):
        shutil.copy(img_file, "/content/yolo_dataset/images/val")
    else:
        print("Missing val image:", img_file)

Copying val images: 100%|██████████| 149/149 [00:06<00:00, 22.49it/s]


In [17]:
# -----------------------------
# 5) YOLO YAML 생성
# -----------------------------
def create_yolo_yaml(yolo_root_dir, train_dir, val_dir, num_classes, class_names):
    data_yaml = {
        "train": train_dir,
        "val": val_dir,
        "nc": num_classes,
        "names": class_names
    }
    yaml_path = os.path.join(yolo_root_dir, "dataset.yaml")
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f, sort_keys=False)
    print(f"✅ YOLO YAML 생성 완료: {yaml_path}")
    return yaml_path

# -----------------------------
# 6) 통합 실행 예시
# -----------------------------
TRAIN_JSON_PATH = "/content/train.json"
TRAIN_IMG_DIR = "/content/train_images"
YOLO_ROOT_DIR = "/content/yolo_dataset"

# 라벨 저장 폴더
TRAIN_LABEL_DIR = os.path.join(YOLO_ROOT_DIR, "labels/train")
VAL_LABEL_DIR   = os.path.join(YOLO_ROOT_DIR, "labels/val")

# 이미지 폴더 (YOLO 학습용)
TRAIN_IMG_YOLO_DIR = os.path.join(YOLO_ROOT_DIR, "images/train")
VAL_IMG_YOLO_DIR   = os.path.join(YOLO_ROOT_DIR, "images/val")

 

# # 2) Train/Val split
# train_df, val_df = split_train_val(df)

# 3) 클래스 매핑
orig2model_yolo, model2orig_yolo, num_classes, unique_cats = build_class_mappings(df)

# 4) 라벨 생성
save_yolo_labels_fast(train_df, TRAIN_LABEL_DIR, orig2model_yolo)
save_yolo_labels_fast(val_df, VAL_LABEL_DIR, orig2model_yolo)

# 5) YAML 생성
CLASS_NAMES = [f"class{i}" for i in range(num_classes)]  # 필요시 실제 클래스명 리스트로 변경
create_yolo_yaml(YOLO_ROOT_DIR, TRAIN_IMG_YOLO_DIR, VAL_IMG_YOLO_DIR, num_classes, CLASS_NAMES)

✅ 클래스 수: 73


Saving labels to /content/yolo_dataset/labels/train: 100%|██████████| 1340/1340 [00:01<00:00, 814.18it/s]
Saving labels to /content/yolo_dataset/labels/val: 100%|██████████| 149/149 [00:00<00:00, 758.18it/s]


✅ YOLO YAML 생성 완료: /content/yolo_dataset/dataset.yaml


'/content/yolo_dataset/dataset.yaml'

In [18]:
!pip install ultralytics

In [19]:
#=============================
# 7. 모델 실행
#=============================
from ultralytics import YOLO

yaml_path = "/content/yolo_dataset/dataset.yaml"  #  YAML path

model = YOLO("yolo11s.pt")   # 여기에서 YOLOlls or YOLO11m ...등 다양한 모델들을 아래 para들과 함께 변경해가면서 적용해보시면 됩니다.

model.train(
    data= yaml_path,
    epochs=10,       # epoch 20, 50 ....
    imgsz=1280,      
    batch=4,        # 16, 32 ....
    device=0,        # GPU memory
    optimizer='AdamW', 
    lr0=1e-3,       # 1e-4... 
    exist_ok=True,
    # augment=True     #  augment가 여기 들어갑니다. 
)

Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, perspect

KeyboardInterrupt: 

In [ ]:
#================
# validation 평가
#================
metrics = model.val(data=yaml_path)

print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision:{metrics.box.mp:.3f}")
print(f"Recall:   {metrics.box.mr:.3f}")

In [ ]:
#==================================
# 8. TEST 평가 및 CSV 생성.
#==================================
import os
import pandas as pd

def yolo_test_inference_ultralytics(model, TEST_IMG_DIR, extract_path, model2orig_yolo, conf_thresh=0.25, top_k=4):
    model.eval()
    results = model.predict(source=TEST_IMG_DIR, imgsz=1280, conf=conf_thresh)

    rows = []

    for r in results:
        image_id = os.path.splitext(os.path.basename(r.path))[0]

        for box in r.boxes:
            cls_id = int(box.cls[0])

            final_category_id = int(model2orig_yolo[cls_id])

            x1, y1, x2, y2 = box.xyxy[0].tolist()

            rows.append({
                "image_id": image_id,
                "category_id": final_category_id,
                "bbox_x": x1,
                "bbox_y": y1,
                "bbox_w": x2 - x1,
                "bbox_h": y2 - y1,
                "score": float(box.conf[0])
            })

    df_sub = pd.DataFrame(rows)

    df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
    df_sub = df_sub.groupby("image_id").head(top_k)

    # 여기서 annotation_id 생성
    df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))

    output_path = os.path.join(extract_path, "final_submission.csv")
    df_sub.to_csv(output_path, index=False)

    print(f"✅ 제출 파일 생성 완료: {output_path}")
    print(f"📊 총 예측 객체 수: {len(df_sub)}")
    print(df_sub.head())

    return df_sub

In [ ]:
model = YOLO("/content/runs/detect/train/weights/best.pt")
df_sub = yolo_test_inference_ultralytics(model, TEST_IMG_DIR, extract_path, model2orig_yolo, conf_thresh=0.25, top_k=4)

In [ ]:
df_sub 